# ЛР4: GQA, KV-cache и инференс

Ноутбук рассчитан на Google Colab. Рекомендуемый runtime: `T4 GPU`.

Что делает notebook:
- клонирует ветку `lab4`;
- ставит зависимости;
- готовит данные на `wikitext-103-raw-v1`;
- проверяет GQA/KV-cache тестами;
- обучает модель с Grouped Query Attention;
- выбирает лучший checkpoint по минимальной `val_perplexity`;
- сравнивает генерацию с KV-cache и без него.

Практическая цель: `val_perplexity <= 25` по заданию. Для более внятного текста лучше дождаться значения ближе к `8-12`.


## 1. Загрузка проекта


In [1]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/BogdanRoshchupkin/modern-ai-architecture.git"
BRANCH = "lab4"
REPO_DIR = Path("/content/modern-ai-architecture")

if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        raise RuntimeError(f"{REPO_DIR} exists, but it is not a git repository")
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
else:
    os.chdir(REPO_DIR)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
print("Active branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("Latest commit:", subprocess.check_output(["git", "log", "--oneline", "-1"], text=True).strip())


Working directory: /content/modern-ai-architecture
Active branch: lab4
Latest commit: 8fc5f31 Validate tokenizer fingerprint for generation


## 2. Установка зависимостей


In [2]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies are ready")


Dependencies are ready


## 3. Проверка GPU и создание `.env`


In [3]:
from pathlib import Path
import subprocess
import torch

try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as exc:
    print("nvidia-smi is not available:", exc)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Path(".env").write_text(f"ROOT_DIR={Path.cwd()}", encoding="utf-8")
print(Path(".env").read_text())


Sat Jun 20 11:30:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 4. Подготовка данных

Для Colab готовим компактный pipeline на Wikitext: clean records -> BPE tokenizer -> packed dataset. Если файлы уже есть, ячейка их не пересоздает.


In [4]:
from pathlib import Path
import subprocess
import sys

processed = Path("data/processed")
processed.mkdir(parents=True, exist_ok=True)

# Для ЛР4 лучше не использовать маленький wikitext-2: на нем GQA-модель быстро упирается
# в плохую validation perplexity. wikitext-103 дает больше разнообразия и ближе к setup ЛР2.
FORCE_REBUILD_DATA = True
WIKITEXT_NAME = "wikitext-103-raw-v1"
WIKITEXT_LIMIT = "50000"

if FORCE_REBUILD_DATA:
    for artifact in [
        "data/processed/wikitext_raw.jsonl",
        "data/processed/wikitext_clean.jsonl",
        "data/processed/common_crawl_bpe.json",
        "data/processed/wikitext_packed.jsonl",
    ]:
        artifact_path = Path(artifact)
        if artifact_path.exists():
            artifact_path.unlink()
            print("removed", artifact)


def run_step(command):
    print("$", " ".join(command))
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print("--- stdout ---")
        print(result.stdout)
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr)
    result.check_returncode()

if not Path("data/processed/wikitext_clean.jsonl").exists():
    run_step([
        sys.executable, "-m", "cli.lab1", "prepare-wikitext",
        "--name", WIKITEXT_NAME,
        "--limit", WIKITEXT_LIMIT,
        "--raw-output", "data/processed/wikitext_raw.jsonl",
        "--clean-output", "data/processed/wikitext_clean.jsonl",
        "--min-words", "5",
    ])
else:
    print("wikitext_clean.jsonl already exists")

if not Path("data/processed/common_crawl_bpe.json").exists():
    run_step([
        sys.executable, "-m", "cli.lab1", "tokenize",
        "--input", "data/processed/wikitext_clean.jsonl",
        "--bpe-vocab-size", "1000",
        "--bpe-output", "data/processed/common_crawl_bpe.json",
        "--bpe-train-limit", WIKITEXT_LIMIT,
    ])
else:
    print("common_crawl_bpe.json already exists")

if not Path("data/processed/wikitext_packed.jsonl").exists():
    run_step([
        sys.executable, "-m", "cli.lab1", "pack",
        "--input", "data/processed/wikitext_clean.jsonl",
        "--tokenizer", "data/processed/common_crawl_bpe.json",
        "--output", "data/processed/wikitext_packed.jsonl",
        "--max-length", "512",
    ])
else:
    print("wikitext_packed.jsonl already exists")

for artifact in [
    "data/processed/wikitext_clean.jsonl",
    "data/processed/common_crawl_bpe.json",
    "data/processed/wikitext_packed.jsonl",
]:
    artifact_path = Path(artifact)
    print(artifact, "exists=", artifact_path.exists(), "size=", artifact_path.stat().st_size if artifact_path.exists() else 0)


$ /usr/bin/python3 -m cli.lab1 prepare-wikitext --name wikitext-103-raw-v1 --limit 50000 --raw-output data/processed/wikitext_raw.jsonl --clean-output data/processed/wikitext_clean.jsonl --min-words 5
--- stdout ---
Downloaded wikitext records: 50000 -> data/processed/wikitext_raw.jsonl
Cleaned wikitext records: 35588 -> data/processed/wikitext_clean.jsonl

--- stderr ---

Generating test split: 100%|██████████| 4358/4358 [00:00<00:00, 92812.52 examples/s]

Generating train split: 100%|██████████| 1801350/1801350 [00:03<00:00, 585316.83 examples/s]

Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 487830.46 examples/s]

$ /usr/bin/python3 -m cli.lab1 tokenize --input data/processed/wikitext_clean.jsonl --bpe-vocab-size 1000 --bpe-output data/processed/common_crawl_bpe.json --bpe-train-limit 50000
--- stdout ---
Char tokenizer vocab size: 1244
Char tokenizer random object length: 725
Word tokenizer vocab size: 94787
Word tokenizer random object length: 132
Saved BPE

## 5. Диагностика подготовленных артефактов

Эта ячейка проверяет, что tokenizer, packed dataset и YAML-конфиг согласованы. Если здесь мало packed rows или vocab mismatch, обучение будет плохим.


In [5]:
import json
from pathlib import Path
from omegaconf import OmegaConf

config = OmegaConf.load("configs/lab4_gqa.yaml")
with Path("data/processed/common_crawl_bpe.json").open("r", encoding="utf-8") as source:
    tokenizer_payload = json.load(source)
packed_path = Path("data/processed/wikitext_packed.jsonl")
with packed_path.open("r", encoding="utf-8") as source:
    first_row = json.loads(next(source))
packed_rows = sum(1 for _ in packed_path.open("r", encoding="utf-8"))
print("tokenizer vocab size:", len(tokenizer_payload["vocab"]))
print("config vocab size:", config.model.vocab_size)
print("packed rows:", packed_rows)
print("first input length:", len(first_row["input_ids"]))
print("first segment ids:", sorted(set(first_row["attention_mask"])))
assert len(tokenizer_payload["vocab"]) == int(config.model.vocab_size)
assert packed_rows >= 1000, "Too few packed rows; increase Wikitext limit or check data preparation"
assert len(first_row["input_ids"]) == int(config.model.max_seq_len)


tokenizer vocab size: 1000
config vocab size: 1000
packed rows: 43094
first input length: 512
first segment ids: [1]


## 6. Проверка GQA и KV-cache тестами


In [6]:
!python -m pytest tests/test_gqa_kv_cache.py -q


..                                                                       [100%]
2 passed in 2.37s


## 7. Быстрая проверка train loop

Эта ячейка запускает один batch и нужна перед полноценным обучением.


In [ ]:
!python -m cli.lab4 train --config configs/lab4_gqa.yaml --fast-dev-run


## 8. Полное обучение

Цель PDF: обучить модель и получить validation perplexity. Для полного балла нужен `val_perplexity <= 25`, для частичного `<= 40`.

Перед полным обучением следующая ячейка очищает старые checkpoint/logs именно для ЛР4. Это важно: иначе notebook может найти старый `final-epoch=00...ckpt`, и инференс будет выглядеть плохо даже после нового запуска.


In [7]:
from pathlib import Path
import shutil

FORCE_RESTART_TRAINING = True

if FORCE_RESTART_TRAINING:
    for artifact_dir in [Path("checkpoints/lab4_gqa"), Path("logs/tensorboard/gqa_kv_cache")]:
        if artifact_dir.exists():
            shutil.rmtree(artifact_dir)
            print("removed", artifact_dir)
else:
    print("Keeping previous checkpoints/logs")


Теперь запускаем полноценное обучение. На T4 это может занять заметное время. Следи за `val_perplexity`: если она падает несколько эпох подряд, обучение идет нормально; если после первой эпохи только растет, лучше остановить runtime и использовать лучший уже сохраненный checkpoint.


In [8]:
!python -m cli.lab4 train --config configs/lab4_gqa.yaml


Выходные данные были обрезаны до нескольких последних строк (5000).
                                                               3.848 train_loss:
                                                               1.434            
                                                               train_perplexity:
Epoch 3/19 ━━╸━━━━━━━━━━━━━ 781/4849 0:01:28 •        8.81it/s v_num: 0.000     
                                     0:07:42                   train_loss_step: 
                                                               1.358 val_loss:  
                                                               1.348            
                                                               val_perplexity:  
                                                               3.848 train_loss:
                                                               1.434            
                                                               train_perplexity:
Epoch 3/19 ━━╸━━━━━━━━━━━━━ 782/4849 0:01

## 9. TensorBoard


In [9]:
%load_ext tensorboard
%tensorboard --logdir logs/tensorboard


<IPython.core.display.Javascript object>

## 10. Найти лучший checkpoint


In [12]:
from pathlib import Path
import math
import re

ckpts = sorted(Path("checkpoints/lab4_gqa").glob("final*.ckpt"))
print("Found checkpoints:")
for ckpt in ckpts:
    print(ckpt)
assert ckpts, "No final checkpoint found. Run training first."

def checkpoint_perplexity(path: Path) -> float:
    match = re.search(r"val_perplexity=([0-9]+(?:\.[0-9]+)?)\.ckpt$", path.name)
    return float(match.group(1)) if match else math.inf

BEST_CKPT = str(min(ckpts, key=checkpoint_perplexity))
print("BEST_CKPT=", BEST_CKPT)
print("BEST_VAL_PERPLEXITY=", checkpoint_perplexity(Path(BEST_CKPT)))
assert checkpoint_perplexity(Path(BEST_CKPT)) <= 25, "Perplexity is above the full-credit target; continue or tune training."


Found checkpoints:
checkpoints/lab4_gqa/final-epoch=02-val_perplexity=3.85.ckpt
BEST_CKPT= checkpoints/lab4_gqa/final-epoch=02-val_perplexity=3.85.ckpt
BEST_VAL_PERPLEXITY= 3.85


## 11. Генерация с KV-cache

Для маленькой модели лучше начинать с более спокойной генерации: `temperature=0.6`, `top_k=10`. Если поставить температуру выше, модель начинает охотнее выбирать редкие BPE-фрагменты, и текст быстро становится шумным.


In [13]:
prompt = "The history of artificial intelligence"
!python -m cli.lab4 generate --config configs/lab4_gqa.yaml --checkpoint "$BEST_CKPT" --prompt "$prompt" --max-new-tokens 80 --temperature 0.6 --top-k 10


The history of artificial intelligence of the program and control to experience a construction to adventure the former


## 12. Генерация без KV-cache для сравнения

Текст должен быть почти таким же по качеству, но генерация обычно медленнее, потому что модель каждый раз заново пересчитывает весь prompt и уже сгенерированные токены.


In [14]:
prompt = "The history of artificial intelligence"
!python -m cli.lab4 generate --config configs/lab4_gqa.yaml --checkpoint "$BEST_CKPT" --prompt "$prompt" --max-new-tokens 80 --temperature 0.6 --top-k 10 --no-kv-cache


The history of artificial intelligence of the episode of the war . The major competition was the first converts were al


## 13. Диагностика распределения следующего токена

Если текст все еще выглядит странно, эта ячейка показывает, какие токены модель считает самыми вероятными после prompt. Если наверху стоят случайные символы или один токен доминирует слишком сильно, проблема в качестве обученного checkpoint, а не в KV-cache.


In [16]:
import torch
from cli.lab2 import load_config
from src.training.lightning_module import GPTLightningModule
from src.tokenization.bpe import BpeTokenizer

config = load_config("configs/lab4_gqa.yaml")
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = BpeTokenizer.load(config.paths.tokenizer)
model = GPTLightningModule.load_from_checkpoint(BEST_CKPT, config=config).to(device)
model.eval()

prompt = "The history of artificial intelligence"
input_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
segment_ids = torch.ones_like(input_ids)
with torch.no_grad():
    logits = model(input_ids, segment_ids)[:, -1, :]
    probs = torch.softmax(logits, dim=-1)
    values, indices = torch.topk(probs, k=20, dim=-1)

inverse_vocab = {idx: token for token, idx in tokenizer.stoi.items()}
print("Top next-token probabilities:")
for value, index in zip(values[0].tolist(), indices[0].tolist()):
    print(f"{index:4d} {inverse_vocab.get(index, '<unk>')!r:18s} {value:.4f}")


Top next-token probabilities:
  81 'o'                0.1626
  75 'i'                0.1350
  89 'w'                0.1180
  67 'a'                0.1008
  13 ','                0.0912
  86 't'                0.0641
  72 'f'                0.0431
  69 'c'                0.0379
  68 'b'                0.0335
  85 's'                0.0259
  84 'r'                0.0182
  70 'd'                0.0174
  82 'p'                0.0172
  74 'h'                0.0156
  79 'm'                0.0134
  15 '.'                0.0131
   9 '('                0.0127
  71 'e'                0.0105
  73 'g'                0.0079
  78 'l'                0.0075


## 14. Сохранение результатов в Google Drive

Эта ячейка монтирует Drive и копирует checkpoints, TensorBoard logs и конфиг. Можно пропустить, если Drive не нужен.


In [17]:
from pathlib import Path
import shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
    target = Path('/content/drive/MyDrive/modern-ai-architecture-lab4')
    target.mkdir(parents=True, exist_ok=True)
    for source in [
        Path('checkpoints/lab4_gqa'),
        Path('logs/tensorboard'),
        Path('configs/lab4_gqa.yaml'),
        Path('data/processed/common_crawl_bpe.json'),
    ]:
        destination = target / source.name
        if source.is_dir():
            if destination.exists():
                shutil.rmtree(destination)
            shutil.copytree(source, destination)
        elif source.exists():
            shutil.copy2(source, destination)
    print('Saved artifacts to', target)
except Exception as exc:
    print('Drive save skipped:', exc)


Mounted at /content/drive
Saved artifacts to /content/drive/MyDrive/modern-ai-architecture-lab4


In [18]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

target = Path("/content/drive/MyDrive/modern-ai-architecture-lab4")
target.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    "data/processed/common_crawl_bpe.json",
    target / "common_crawl_bpe.json",
)

print("saved to:", target / "common_crawl_bpe.json")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saved to: /content/drive/MyDrive/modern-ai-architecture-lab4/common_crawl_bpe.json


## 15. Что показать на защите

- В `configs/lab4_gqa.yaml`: `n_heads=8`, `n_kv_heads=4`, это Grouped Query Attention.
- В тестах: cached incremental logits совпадают с full forward logits.
- В TensorBoard/output обучения: лучший `val_perplexity`.
- В генерации: пример с KV-cache и пример с `--no-kv-cache`.
- В диагностике top-token'ов: модель не должна уверенно выбирать случайные BPE-обрывки после обычного prompt.
